In [2]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.interpolate import interp1d
from scipy.linalg import solve_banded
from scipy.integrate import solve_ivp



In [3]:
# ------------------------------------construccion de la malla

M = 1
                       #Numero de puntos de la malla
rmin = 0
rmax = 10
N= 1000                                #Numero de puntos en la malla
delta_r = (rmax-rmin)/N                 #distancia al primer punto de la malla

# ri= rmin +(i-1/2) delta_r                   define el punto iesimo

points_r_interior = np.arange(rmin + 1/2*delta_r , rmax, delta_r)
ghost_point = 2       # 2 por cada frontera

ghost_left = np.array([rmin - (3/2)*delta_r,rmin -(1/2)*delta_r])
ghost_right = np.array([rmax + (1/2)*delta_r,rmax + (3/2)*delta_r])
points_r= np.concatenate((ghost_left,points_r_interior,ghost_right))

N_ghost= len(points_r)


# Condiciones iniciales ------------------------------------
alpha  = np.ones(N)
Atilde = np.ones(N)
Btilde = np.ones(N)
KA     = np.zeros(N)
KB     = np.zeros(N)
DAtilde = np.zeros(N)
DBtilde = np.zeros(N)

psi_full = 1.0 + M / (2 *np.abs(points_r))
lnpsi = np.log(np.abs(psi_full))
dr_lnpsi_full= -M/(np.abs(points_r)*(2*np.abs(points_r) + M))

psi=psi_full[2:-2]
dr_lnpsi = dr_lnpsi_full[2:-2]


In [4]:
def agregar_fantasmas(f_fisico, paridad, dr, r_fisico=points_r_interior, bc_outer="robin", c_robin=1.0):
    N = len(f_fisico)
    f_full = np.empty(N + 4)

    # 1. Copiar el dominio físico al centro
    f_full[2:-2] = f_fisico

    # 2. FRONTERA INTERIOR (r=0, malla sin incluir el origen exacto)
    # f_full[2] está en +dr/2; su reflejo f_full[1] está en -dr/2
    f_full[1] = paridad * f_full[2]
    f_full[0] = paridad * f_full[3]

    # 3. FRONTERA EXTERIOR (r = r_max)
    if bc_outer == "extrapolacion":
        # Extrapolación lineal (mantiene la primera derivada)
        f_full[-2] = 2.0 * f_full[-3] - f_full[-4]
        f_full[-1] = 2.0 * f_full[-2] - f_full[-3]

    elif bc_outer == "robin":
        # Robin: df/dr + (c / r) * f = 0  => df/dr = - (c / r) * f
        # Despejando f_{i+1} con diferencia centrada: f_{i+1} = f_{i-1} - 2*dr*(c/r)*f_i
        if r_fisico is None:
            raise ValueError("Debes proporcionar 'r_fisico' para usar la frontera de Robin.")

        r_max = r_fisico[-1]
        k = c_robin / r_max  # Coeficiente de decaimiento en la frontera

        # Primer fantasma exterior (índice -2)
        f_full[-2] = f_full[-4] - 2.0 * dr * k * f_full[-3]
        # Segundo fantasma exterior (índice -1)
        f_full[-1] = f_full[-3] - 2.0 * dr * k * f_full[-2]

    else:
        raise ValueError(f"Condición de frontera '{bc_outer}' no válida.")

    return f_full

def D1_4_order(var_fisica, dx, paridad=1, bc_type="extrapolacion", bc_value=0.0):
    N = len(var_fisica)
    f = agregar_fantasmas(var_fisica,paridad,dx)

    # Devuelve exactamente un arreglo de tamaño N sin gastar memoria extra
    df_fisico = (-f[4:] + 8*f[3:-1] - 8*f[1:-3] + f[:-4]) / (12 * dx)

    return df_fisico

In [ ]:
# def solve_lapse_penta(Atilde, DBtilde, DAtilde, KA, KB, psi, dlnpsi1, r, dr):
#     N=len(r)
#     P = 2.0/r + DBtilde - 0.5*DAtilde + 2.0*dlnpsi1
#     Q = Atilde * (psi**4) * (KA**2 + 2.0*KB**2)

#     ab = np.zeros((5, N))
#     rhs = np.zeros(N)


#     # 1. PUNTOS INTERIORES (i = 2 hasta N-3) ignoramos puntos fantasma
#     for i in range(2, N-2):
#         c_l2 = (-1.0 / (12*dr**2)) + P[i] * ( 1.0 / (12*dr))
#         c_l1 = (16.0 / (12*dr**2)) + P[i] * (-8.0 / (12*dr))
#         c_0  = (-30.0/ (12*dr**2))
#         c_r1 = (16.0 / (12*dr**2)) + P[i] * ( 8.0 / (12*dr))
#         c_r2 = (-1.0 / (12*dr**2)) + P[i] * (-1.0 / (12*dr))

#         ab[4, i-2] = c_l2  # j = i-2
#         ab[3, i-1] = c_l1  # j = i-1
#         ab[2, i]   = c_0   # j = i
#         ab[1, i+1] = c_r1  # j = i+1
#         ab[0, i+2] = c_r2  # j = i+2
#         rhs[i] = Q[i]

#     # 2. FRONTERA INTERIOR (r=0)
#     # alpha_1 - alpha_2 = 0
#     ab[2, 1] =  1.0  # i=0, j=1
#     ab[1, 2] = -1.0  # i=1, j=2
#     rhs[1]   =  0.0

#     # alpha_0 - alpha_1 = 0
#     ab[2, 0] =  1.0  # i=0, j=0
#     ab[1, 1] = -1.0  # i=0, j=1
#     rhs[0]   =  0.0

#     # 3. FRONTERA EXTERIOR (Robin) en i = N-2
#     i_ext = N-2
#     ab[3, i_ext-1] = -1.0 / (2*dr)       # j = N-3
#     ab[2, i_ext]   =  1.0 / r[i_ext] # j = N-2
#     ab[1, i_ext+1] =  1.0 / (2*dr)       # j = N-1
#     rhs[i_ext]     =  1.0 / r[i_ext]

#     # Fantasma extremo exterior plano (alpha_{N-1} - alpha_{N-2} = 0)
#     ab[2, N-1] =  1.0  # i=N-1, j=N-1
#     ab[3, N-2] = -1.0  # i=N-1, j=N-2
#     rhs[N-1]   =  0.0

#     # 4. RESOLVER EL SISTEMA
#     alpha = solve_banded((2, 2), ab, rhs)

#     return alpha

In [6]:
def solve_lapse_penta(Atilde, DBtilde, DAtilde, KA, KB, psi, dlnpsi1, r, dr):
    N = len(r)
    P = 2.0/r + DBtilde - 0.5*DAtilde + 2.0*dlnpsi1
    Q = Atilde * (psi**4) * (KA**2 + 2.0*KB**2)

    ab = np.zeros((6, N))
    rhs = np.zeros(N)

    # 1. PUNTOS INTERIORES (i = 2 hasta N-3)
    for i in range(2, N-2):
        c_l2 = -1.0  + P[i] * (dr)
        c_l1 = 16.0  + P[i] * (-8.0 *dr)
        c_0  = -30.0
        c_r1 = 16.0  + P[i] * ( 8.0 *dr)
        c_r2 = -1.0  + P[i] * (-1.0 *dr)

        # Fila en la matriz bandeada = u + i - j  (con u = 3)
        ab[5, i-2] = c_l2  # j = i-2 -> row = 3 + i - (i-2) = 5
        ab[4, i-1] = c_l1  # j = i-1 -> row = 3 + i - (i-1) = 4
        ab[3, i]   = c_0   # j = i   -> row = 3 + i - i = 3
        ab[2, i+1] = c_r1  # j = i+1 -> row = 3 + i - (i+1) = 2
        ab[1, i+2] = c_r2  # j = i+2 -> row = 3 + i - (i+2) = 1

        rhs[i] = Q[i]*12*dr**2

    # 2. FRONTERA INTERIOR (r=0)
    # alpha_1 - alpha_2 = 0
    ab[3, 1] =  1.0  # i=1, j=1 -> row = 3 + 1 - 1 = 3
    ab[2, 2] = -1.0  # i=1, j=2 -> row = 3 + 1 - 2 = 2
    rhs[1]   =  0.0

    # alpha_0 - alpha_3 = 0
    ab[3, 0] =  1.0  # i=0, j=0 -> row = 3 + 0 - 0 = 3
    ab[0, 3] = -1.0  # i=0, j=3 -> row = 3 + 0 - 3 = 0
    rhs[0]   =  0.0

    # 3. FRONTERA EXTERIOR (Robin) en i = N-2
    i_ext = N-2
    ab[4, i_ext-1] = -1.0 / (2*dr)       # j = N-3 -> row = 4
    ab[3, i_ext]   =  1.0 / r[i_ext]     # j = N-2 -> row = 3
    ab[2, i_ext+1] =  1.0 / (2*dr)       # j = N-1 -> row = 2
    rhs[i_ext]     =  1.0 / r[i_ext]

    # Fantasma extremo exterior plano (alpha_{N-1} - alpha_{N-2} = 0)
    ab[3, N-1] =  1.0  # i=N-1, j=N-1 -> row = 3
    ab[4, N-2] = -1.0  # i=N-1, j=N-2 -> row = 4
    rhs[N-1]   =  0.0

    # 4. RESOLVER EL SISTEMA (l=2, u=3)

    alpha = solve_banded((2, 3), ab, rhs)

    return alpha


In [7]:
def KO(u_fisico, dx, paridad=1, bc_outer="robin", eps=0.2):
    """
    Método de Disipación de Kreiss-Oliger de 4to orden.
    Recibe un arreglo de tamaño N, usa fantasmas temporalmente y retorna tamaño N.
    """
    # 1. Inflar el arreglo a tamaño N+4
    u_full = agregar_fantasmas(u_fisico, paridad=paridad, dr=dx, bc_outer=bc_outer)

    # 2. Aplicar el stencil sobre u_full
    # Al hacer esto, 'q' sale matemáticamente con el tamaño exacto N (1000)
    q = -eps*(u_full[:-4] - 4*u_full[1:-3] + 6*u_full[2:-2] - 4*u_full[3:-1] + u_full[4:]) / (16*dx)

    return q

def RHS(t,y_global,N,r,dr=delta_r):

  A_tilde= y_global[0:N]
  B_tilde= y_global[N:2*N]
  DA_tilde= y_global[2*N:3*N]
  DB_tilde= y_global[3*N:4*N]
  KA= y_global[4*N:5*N]
  KB= y_global[5*N:6*N]


  Atilde_full  = agregar_fantasmas(A_tilde,  paridad=1,  dr=dr, bc_outer="extrapolacion")
  DBtilde_full = agregar_fantasmas(DB_tilde, paridad=-1, dr=dr, bc_outer="extrapolacion")
  DAtilde_full = agregar_fantasmas(DA_tilde, paridad=-1, dr=dr, bc_outer="extrapolacion")
  KA_full      = agregar_fantasmas(KA,      paridad=1,  dr=dr, bc_outer="extrapolacion")
  KB_full      = agregar_fantasmas(KB,      paridad=1,  dr=dr, bc_outer="extrapolacion")


  alpha_full = solve_lapse_penta(
        Atilde_full, DBtilde_full, DAtilde_full,
        KA_full, KB_full, psi_full, dr_lnpsi_full, points_r, dr )

  alpha= alpha_full[2:-2]


  Dalpha= D1_4_order(alpha,dr,paridad=1)/alpha

  dKA = D1_4_order(KA,dr,paridad=1)
  dKB = D1_4_order(KB,dr,paridad=1)
  dDB = D1_4_order(DB_tilde,dr,paridad=-1)
  dDalpha = D1_4_order(Dalpha,dr,paridad=-1)

  ddlnpsi = D1_4_order(dr_lnpsi,dr,paridad=-1)

  # sistema de ecuaciones
  dA_dt = -2*alpha*A_tilde*KA
  dB_dt = -2*alpha*B_tilde*KB

  dDA_dt = -2*alpha*(KA*Dalpha + dKA) - 4*dr_lnpsi
  dDB_dt = -2*alpha*(KB*Dalpha + dKB) - 4*dr_lnpsi


  T1 = dDalpha + dDB + 4*ddlnpsi
  T2 = Dalpha**2
  T3 = -0.5*Dalpha*(DA_tilde + 4*dr_lnpsi)
  T4 = -(1/r)*(DA_tilde+4*dr_lnpsi-2*Dalpha-4*DB_tilde-16*dr_lnpsi)
  T5 = -(2/(B_tilde*r**2))*(A_tilde-B_tilde)

  dKA_dt = -alpha/(psi**4*A_tilde)*(T1+T2+T3+T4+T5)+alpha*KB*(KA+2*KB)


  S1 = dDB + 4*ddlnpsi
  S2 = Dalpha*(DB_tilde + 4*dr_lnpsi)
  S3 = (DB_tilde + 4*dr_lnpsi)**2
  S4 = -0.5*(DA_tilde+4*dr_lnpsi)*(DB_tilde+4*dr_lnpsi)
  S5 = T4
  S6 = T5
  dKB_dt = (-alpha/(2*psi**4*A_tilde)*(S1+S2+S3+S4+S5+S6)+alpha*KB*(KA+2*KB))

  # dA_dt  += KO(A_tilde, dr, paridad=1)
  # dB_dt  += KO(B_tilde, dr, paridad=1)
  # dDA_dt += KO(DA_tilde, dr, paridad=-1)
  # dDB_dt += KO(DB_tilde, dr, paridad=-1)
  # dKA_dt += KO(KA, dr, paridad=1)
  # dKB_dt += KO(KB, dr, paridad=1)

  return np.concatenate([dA_dt,dB_dt,dDA_dt,dDB_dt,dKA_dt,dKB_dt])

In [8]:
delta_t=delta_r*0.25
ti=0
tf=1
t_span=(ti,tf)
y0=np.concatenate([Atilde,Btilde,DAtilde,DBtilde,KA,KB])
t_eval=np.arange(ti,tf,delta_t)

sol = solve_ivp(
    RHS,
    t_span,
    y0,
    t_eval=t_eval,
    args=(N, points_r_interior),
    method="RK45",
    max_step=delta_t
)

/tmp/ipykernel_1582/1853731245.py:49: RuntimeWarning: overflow encountered in multiply
  dA_dt = -2*alpha*A_tilde*KA
/tmp/ipykernel_1582/1853731245.py:50: RuntimeWarning: overflow encountered in multiply
  dB_dt = -2*alpha*B_tilde*KB
/tmp/ipykernel_1582/1853731245.py:62: RuntimeWarning: overflow encountered in multiply
  dKA_dt = -alpha/(psi**4*A_tilde)*(T1+T2+T3+T4+T5)+alpha*KB*(KA+2*KB)
/tmp/ipykernel_1582/1853731245.py:71: RuntimeWarning: overflow encountered in multiply
  dKB_dt = (-alpha/(2*psi**4*A_tilde)*(S1+S2+S3+S4+S5+S6)+alpha*KB*(KA+2*KB))
/tmp/ipykernel_1582/1853731245.py:52: RuntimeWarning: overflow encountered in multiply
  dDA_dt = -2*alpha*(KA*Dalpha + dKA) - 4*dr_lnpsi
/tmp/ipykernel_1582/1853731245.py:53: RuntimeWarning: overflow encountered in multiply
  dDB_dt = -2*alpha*(KB*Dalpha + dKB) - 4*dr_lnpsi


In [ ]:
sol.y